# Views & Triggers

## Quick notes
### What this covers
- Views: saved SELECTs you can query like tables, and how to make them writable.
- Triggers: code that runs automatically on INSERT, UPDATE or DELETE (audit logs, timestamps, validation).
### Key points
- A view stores a query, not data. Every use re-runs the query.
- Views are read-only unless you add `INSTEAD OF` triggers.
- SQLite has no `CREATE OR REPLACE VIEW` and no real materialized views. Drop and recreate, or use a table you refresh.
- A trigger = event (INSERT/UPDATE/DELETE) + timing (BEFORE/AFTER/INSTEAD OF) + optional `WHEN` + a body.
- Inside a trigger, `NEW.col` is the row being written and `OLD.col` is the row before the change.
- `SELECT RAISE(ABORT, 'message')` stops the statement and undoes it.
### Gotchas (the part worth revising twice)
- SQLite triggers are always FOR EACH ROW. There is no per-statement trigger.
- INSERT has no OLD, DELETE has no NEW.
- A trigger that updates its own table can loop forever in other databases. In SQLite it is limited by `recursive_triggers` (OFF by default).
- `INSERT OR REPLACE` deletes rows without firing DELETE triggers unless `recursive_triggers` is ON.
- Triggers hide logic. Too many make the data layer hard to debug; prefer constraints when they can do the job.
- A view with `SELECT *` follows later table changes; a view on a dropped column breaks when used.
### When it matters
- Hiding columns or complex joins behind a simple name.
- Keeping history/audit trails and "last changed" columns automatic, no matter which app writes.
### One-line recap example
```sql
CREATE VIEW v AS SELECT id, name FROM t;
CREATE TRIGGER trg AFTER UPDATE ON t BEGIN INSERT INTO log VALUES (OLD.id, OLD.name, NEW.name); END;
```

## 01. Seed data
- Small company data used throughout: departments and employees with salaries.
- Dates are fixed text so results never change.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE departments (
    dept_id   INTEGER PRIMARY KEY,
    dept_name TEXT NOT NULL
);
CREATE TABLE employees (
    emp_id     INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    dept_id    INTEGER NOT NULL REFERENCES departments (dept_id),
    salary     REAL NOT NULL,
    hired_on   TEXT NOT NULL
);
INSERT INTO departments VALUES (1, 'Engineering'), (2, 'Sales'), (3, 'Marketing');
INSERT INTO employees VALUES
    (1, 'Alice', 1, 95000, '2020-01-15'),
    (2, 'Bob',   1, 72000, '2021-03-01'),
    (3, 'Carla', 2, 68000, '2019-07-20'),
    (4, 'Dev',   2, 61000, '2022-11-05'),
    (5, 'Eve',   3, 58000, '2023-02-10');
SELECT emp_id, first_name, dept_id, salary FROM employees ORDER BY emp_id;

emp_id | first_name | dept_id | salary 
-------+------------+---------+--------
1      | Alice      | 1       | 95000.0
2      | Bob        | 1       | 72000.0
3      | Carla      | 2       | 68000.0
4      | Dev        | 2       | 61000.0
5      | Eve        | 3       | 58000.0
(5 rows)



## 02. CREATE VIEW
- A view gives a name to a SELECT. It holds no data of its own.
- Every time you query it, SQLite runs the saved SELECT on the current table data.
- Good for hiding a join behind a simple, stable name.

In [3]:
%%sql
CREATE VIEW high_earners AS
SELECT e.emp_id, e.first_name, e.salary, d.dept_name
FROM employees e
JOIN departments d ON d.dept_id = e.dept_id
WHERE e.salary > 65000;

SELECT * FROM high_earners ORDER BY salary DESC;

emp_id | first_name | salary  | dept_name  
-------+------------+---------+------------
1      | Alice      | 95000.0 | Engineering
2      | Bob        | 72000.0 | Engineering
3      | Carla      | 68000.0 | Sales      
(3 rows)



## 03. Querying a view like a table
- You can filter, sort, join and aggregate over a view.
- The view is always fresh: change the table and the view changes with it.

In [4]:
%%sql
SELECT first_name, salary FROM high_earners WHERE dept_name = 'Engineering' ORDER BY first_name;
UPDATE employees SET salary = 70000 WHERE first_name = 'Dev';   -- Dev now crosses the 65000 line
SELECT first_name, salary FROM high_earners ORDER BY salary DESC, first_name;

first_name | salary 
-----------+--------
Alice      | 95000.0
Bob        | 72000.0
(2 rows)

first_name | salary 
-----------+--------
Alice      | 95000.0
Bob        | 72000.0
Dev        | 70000.0
Carla      | 68000.0
(4 rows)



## 04. Views for abstraction and simple security
- SQLite has no GRANT, so views cannot restrict users by themselves.
- But a view can hide sensitive columns (salary) from code that only gets the view name, e.g. a reporting tool or an app layer you control.
- It also lets you rename columns and keep a stable "public" shape while tables change underneath.

In [5]:
%%sql
CREATE VIEW staff_directory AS
SELECT emp_id, first_name AS name, dept_id
FROM employees;               -- salary and hired_on are not exposed

SELECT * FROM staff_directory ORDER BY emp_id;

emp_id | name  | dept_id
-------+-------+--------
1      | Alice | 1      
2      | Bob   | 1      
3      | Carla | 2      
4      | Dev   | 2      
5      | Eve   | 3      
(5 rows)



## 05. Views over joins and aggregates
- A view can hide a GROUP BY so analysts just `SELECT * FROM dept_summary`.
- Column names come from the SELECT list, so always give aggregates an alias.
- SUM over zero rows is NULL, so we wrap it in COALESCE to get 0 for an empty department.
- You can also name the columns up front: `CREATE VIEW v (a, b) AS ...`.

In [6]:
%%sql
CREATE VIEW dept_summary (department, head_count, total_salary, avg_salary) AS
SELECT d.dept_name, COUNT(e.emp_id), COALESCE(SUM(e.salary), 0), ROUND(AVG(e.salary), 1)
FROM departments d
LEFT JOIN employees e ON e.dept_id = d.dept_id
GROUP BY d.dept_id, d.dept_name;

SELECT * FROM dept_summary ORDER BY department;

department  | head_count | total_salary | avg_salary
------------+------------+--------------+-----------
Engineering | 2          | 167000.0     | 83500.0   
Marketing   | 1          | 58000.0      | 58000.0   
Sales       | 2          | 138000.0     | 69000.0   
(3 rows)



## 06. Views stacked on views
- A view can be built from another view.
- Handy for layers: raw join -> cleaned view -> report view.
- Keep layers shallow. Deep stacks are hard to read and hard to optimise.

In [7]:
%%sql
CREATE VIEW big_departments AS
SELECT department, head_count FROM dept_summary WHERE head_count >= 2;

SELECT * FROM big_departments ORDER BY department;

department  | head_count
------------+-----------
Engineering | 2         
Sales       | 2         
(2 rows)



## 07. Views vs CTEs
- A CTE (`WITH ...`) exists only inside one statement. A view is saved in the database and reused anywhere.
- Use a CTE for one-off readability inside a query; use a view when many queries share the same logic.
- Both are re-evaluated each time (neither caches results). See 02-Querying/03-Subqueries-And-CTEs.

In [8]:
%%sql
WITH rich AS (
    SELECT dept_id, salary FROM employees WHERE salary > 65000
)
SELECT dept_id, COUNT(*) AS n FROM rich GROUP BY dept_id ORDER BY dept_id;

dept_id | n
--------+--
1       | 2
2       | 2
(2 rows)



## 08. SELECT * in a view and table changes
- A view saved with `SELECT *` picks up columns added to the table later.
- Bad: relying on `*` makes the view's shape drift. Good: list the columns you want.
- Dropping a column a view uses breaks the view only when you next query it.

In [9]:
%%sql
CREATE TABLE gadgets (id INTEGER PRIMARY KEY, name TEXT);
INSERT INTO gadgets VALUES (1, 'Widget');
CREATE VIEW gadgets_all AS SELECT * FROM gadgets;
ALTER TABLE gadgets ADD COLUMN color TEXT DEFAULT 'grey';
SELECT * FROM gadgets_all ORDER BY id;

id | name   | color
---+--------+------
1  | Widget | grey 
(1 row)



## 09. DROP VIEW and the replace pattern
- `DROP VIEW name` removes only the saved query. Tables and data are untouched.
- `DROP VIEW IF EXISTS name` avoids an error if it is missing.
- There is no `CREATE OR REPLACE VIEW`. To change a view: drop it, then create it again (in a transaction if others use it).
- `CREATE VIEW IF NOT EXISTS` does NOT update an existing view; it silently keeps the old one.

In [10]:
%%sql
BEGIN;
DROP VIEW IF EXISTS high_earners;
CREATE VIEW high_earners AS
SELECT e.emp_id, e.first_name, e.salary, d.dept_name
FROM employees e JOIN departments d ON d.dept_id = e.dept_id
WHERE e.salary >= 70000;
COMMIT;
SELECT first_name, salary FROM high_earners ORDER BY first_name;
SELECT COUNT(*) AS employees_untouched FROM employees;

first_name | salary 
-----------+--------
Alice      | 95000.0
Bob        | 72000.0
Dev        | 70000.0
(3 rows)

employees_untouched
-------------------
5                  
(1 row)



## 10. Views are read-only by default
- You cannot INSERT, UPDATE or DELETE through a view.
- To make it writable, add `INSTEAD OF` triggers (next sections).

In [11]:
%%sql expect_error
INSERT INTO staff_directory (name, dept_id) VALUES ('Zoe', 1);
UPDATE staff_directory SET name = 'Al' WHERE emp_id = 1;

ERROR: cannot modify staff_directory because it is a view

ERROR: cannot modify staff_directory because it is a view



## 11. Trigger anatomy
- `CREATE TRIGGER name {BEFORE|AFTER|INSTEAD OF} {INSERT|UPDATE [OF cols]|DELETE} ON table [WHEN condition] BEGIN statements; END;`
- It runs once for every affected row.
- `NEW.col` exists for INSERT and UPDATE. `OLD.col` exists for UPDATE and DELETE.
- `INSTEAD OF` is only for views. `BEFORE`/`AFTER` are for tables.
- The body is one or more statements, each ending with `;`.

## 12. INSTEAD OF triggers: making a view writable
- The trigger catches the write to the view and decides what to do with real tables.
- Here `staff_directory` hides salary, so inserts must supply a default salary.
- The view column is `name`, the table column is `first_name`: the trigger translates.

In [12]:
%%sql
CREATE TRIGGER staff_directory_insert
INSTEAD OF INSERT ON staff_directory
BEGIN
    INSERT INTO employees (first_name, dept_id, salary, hired_on)
    VALUES (NEW.name, NEW.dept_id, 40000, '2024-01-01');
END;

CREATE TRIGGER staff_directory_update
INSTEAD OF UPDATE ON staff_directory
BEGIN
    UPDATE employees SET first_name = NEW.name, dept_id = NEW.dept_id
    WHERE emp_id = OLD.emp_id;
END;

CREATE TRIGGER staff_directory_delete
INSTEAD OF DELETE ON staff_directory
BEGIN
    DELETE FROM employees WHERE emp_id = OLD.emp_id;
END;

INSERT INTO staff_directory (name, dept_id) VALUES ('Zoe', 1);
UPDATE staff_directory SET name = 'Albert' WHERE emp_id = 1;
DELETE FROM staff_directory WHERE name = 'Eve';
SELECT emp_id, first_name, dept_id, salary FROM employees ORDER BY emp_id;

emp_id | first_name | dept_id | salary 
-------+------------+---------+--------
1      | Albert     | 1       | 95000.0
2      | Bob        | 1       | 72000.0
3      | Carla      | 2       | 68000.0
4      | Dev        | 2       | 70000.0
6      | Zoe        | 1       | 40000.0
(5 rows)



## 13. Emulating a materialized view
- A materialized view stores query results so reads are fast. SQLite has none.
- Emulate it: a real table holding the results, plus a refresh step (delete + insert, in a transaction).
- Trade-off: reads are cheap but the data can be stale until you refresh.
- You can refresh from app code, a scheduled job, or a trigger (see 18).

In [13]:
%%sql
CREATE TABLE dept_summary_cache (
    department   TEXT PRIMARY KEY,
    head_count   INTEGER NOT NULL,
    total_salary REAL NOT NULL
);

BEGIN;
DELETE FROM dept_summary_cache;
INSERT INTO dept_summary_cache
SELECT department, head_count, total_salary FROM dept_summary;
COMMIT;

SELECT * FROM dept_summary_cache ORDER BY department;

department  | head_count | total_salary
------------+------------+-------------
Engineering | 3          | 207000.0    
Marketing   | 0          | 0.0         
Sales       | 2          | 138000.0    
(3 rows)



In [14]:
%%sql
UPDATE employees SET salary = salary + 1000 WHERE dept_id = 1;
SELECT 'cache (stale)' AS source, total_salary FROM dept_summary_cache WHERE department = 'Engineering'
UNION ALL
SELECT 'live view', total_salary FROM dept_summary WHERE department = 'Engineering'
ORDER BY source;

source        | total_salary
--------------+-------------
cache (stale) | 207000.0    
live view     | 210000.0    
(2 rows)



## 14. AFTER trigger: audit log
- An audit table records who/what changed and the old and new values.
- `AFTER UPDATE` runs once the row is already changed, so OLD and NEW are both available.
- Use `UPDATE OF salary` or a `WHEN` clause to fire only when it matters.
- Real systems add a timestamp column with `DEFAULT CURRENT_TIMESTAMP`. We skip showing it so output stays fixed.

In [15]:
%%sql
CREATE TABLE salary_audit (
    audit_id   INTEGER PRIMARY KEY,
    emp_id     INTEGER NOT NULL,
    old_salary REAL,
    new_salary REAL,
    action     TEXT NOT NULL,
    changed_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE TRIGGER trg_salary_update
AFTER UPDATE OF salary ON employees
WHEN OLD.salary IS NOT NEW.salary
BEGIN
    INSERT INTO salary_audit (emp_id, old_salary, new_salary, action)
    VALUES (OLD.emp_id, OLD.salary, NEW.salary, 'UPDATE');
END;

CREATE TRIGGER trg_employee_insert
AFTER INSERT ON employees
BEGIN
    INSERT INTO salary_audit (emp_id, old_salary, new_salary, action)
    VALUES (NEW.emp_id, NULL, NEW.salary, 'INSERT');
END;

CREATE TRIGGER trg_employee_delete
AFTER DELETE ON employees
BEGIN
    INSERT INTO salary_audit (emp_id, old_salary, new_salary, action)
    VALUES (OLD.emp_id, OLD.salary, NULL, 'DELETE');
END;

In [16]:
%%sql
UPDATE employees SET salary = 100000 WHERE emp_id = 1;
UPDATE employees SET salary = 100000 WHERE emp_id = 1;     -- no change: WHEN blocks it
UPDATE employees SET first_name = 'Bobby' WHERE emp_id = 2; -- not the salary column: trigger does not fire
INSERT INTO employees VALUES (10, 'Fay', 3, 50000, '2024-06-01');
DELETE FROM employees WHERE emp_id = 10;
SELECT audit_id, emp_id, old_salary, new_salary, action FROM salary_audit ORDER BY audit_id;

audit_id | emp_id | old_salary | new_salary | action
---------+--------+------------+------------+-------
1        | 1      | 96000.0    | 100000.0   | UPDATE
2        | 10     | NULL       | 50000.0    | INSERT
3        | 10     | 50000.0    | NULL       | DELETE
(3 rows)



## 15. updated_at / version maintenance
- A trigger can stamp a row every time it changes, so no app can forget.
- Pattern: `AFTER UPDATE` trigger that runs a second UPDATE on the same row.
- Guard it with `WHEN` so it does not keep re-triggering itself. By default SQLite also stops direct recursion (see 19).
- We keep a `version` counter (deterministic) next to `updated_at`.

In [17]:
%%sql
CREATE TABLE notes (
    note_id    INTEGER PRIMARY KEY,
    body       TEXT NOT NULL,
    version    INTEGER NOT NULL DEFAULT 1,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE TRIGGER trg_notes_touch
AFTER UPDATE OF body ON notes
BEGIN
    UPDATE notes SET version = OLD.version + 1, updated_at = CURRENT_TIMESTAMP
    WHERE note_id = NEW.note_id;
END;

INSERT INTO notes (body) VALUES ('first draft');
UPDATE notes SET body = 'second draft' WHERE note_id = 1;
UPDATE notes SET body = 'final draft' WHERE note_id = 1;
SELECT note_id, body, version, updated_at IS NOT NULL AS stamped FROM notes ORDER BY note_id;

note_id | body        | version | stamped
--------+-------------+---------+--------
1       | final draft | 3       | 1      
(1 row)



## 16. Validation with RAISE(ABORT, ...)
- A `BEFORE` trigger can check a rule that a CHECK cannot (it may look at other tables or rows) and reject the change.
- `SELECT RAISE(ABORT, 'message')` cancels the statement, undoes its changes, and returns your message as the error.
- Rule here: a raise may not exceed 20% of the old salary, and salary may never drop below 30000.

In [18]:
%%sql
CREATE TRIGGER trg_salary_rules
BEFORE UPDATE OF salary ON employees
BEGIN
    SELECT RAISE(ABORT, 'salary below minimum 30000')
    WHERE NEW.salary < 30000;
    SELECT RAISE(ABORT, 'raise larger than 20 percent')
    WHERE NEW.salary > OLD.salary * 1.2;
END;

In [19]:
%%sql expect_error
UPDATE employees SET salary = 20000 WHERE emp_id = 3;
UPDATE employees SET salary = 99999 WHERE emp_id = 3;

ERROR: salary below minimum 30000

ERROR: raise larger than 20 percent



In [20]:
%%sql
UPDATE employees SET salary = 70000 WHERE emp_id = 3;
SELECT emp_id, salary FROM employees WHERE emp_id = 3 ORDER BY emp_id;

emp_id | salary 
-------+--------
3      | 70000.0
(1 row)



## 17. The RAISE variants
- `RAISE(ABORT, msg)`: stop this statement, undo it, keep earlier statements of the transaction. Most common.
- `RAISE(ROLLBACK, msg)`: abort and roll back the whole transaction.
- `RAISE(FAIL, msg)`: stop, but keep changes already made by this statement (rows processed before the failure stay).
- `RAISE(IGNORE)`: silently skip this row (and the rest of this trigger program); no error.
- Demo of IGNORE: a BEFORE DELETE trigger that protects one row without any error.

In [21]:
%%sql
CREATE TABLE settings (key TEXT PRIMARY KEY, value TEXT NOT NULL, locked INTEGER NOT NULL DEFAULT 0);
INSERT INTO settings VALUES ('theme', 'dark', 0), ('license', 'abc', 1);

CREATE TRIGGER trg_settings_protect
BEFORE DELETE ON settings
WHEN OLD.locked = 1
BEGIN
    SELECT RAISE(IGNORE);
END;

DELETE FROM settings;
SELECT * FROM settings ORDER BY key;

key     | value | locked
--------+-------+-------
license | abc   | 1     
(1 row)



## 18. Keeping a summary table in sync with triggers
- Triggers can maintain a running total so reads never need a GROUP BY.
- Bad: easy to drift if some write path bypasses it, but triggers sit inside the database so every writer is covered.
- Here: per-department head count, updated on insert, delete, and department change.

In [22]:
%%sql
CREATE TABLE dept_headcount (dept_id INTEGER PRIMARY KEY, n INTEGER NOT NULL);
INSERT INTO dept_headcount SELECT dept_id, COUNT(*) FROM employees GROUP BY dept_id;

CREATE TRIGGER trg_hc_ins AFTER INSERT ON employees
BEGIN
    INSERT INTO dept_headcount VALUES (NEW.dept_id, 1)
    ON CONFLICT (dept_id) DO UPDATE SET n = n + 1;
END;
CREATE TRIGGER trg_hc_del AFTER DELETE ON employees
BEGIN
    UPDATE dept_headcount SET n = n - 1 WHERE dept_id = OLD.dept_id;
END;
CREATE TRIGGER trg_hc_move AFTER UPDATE OF dept_id ON employees
WHEN OLD.dept_id <> NEW.dept_id
BEGIN
    UPDATE dept_headcount SET n = n - 1 WHERE dept_id = OLD.dept_id;
    INSERT INTO dept_headcount VALUES (NEW.dept_id, 1)
    ON CONFLICT (dept_id) DO UPDATE SET n = n + 1;
END;

INSERT INTO employees VALUES (11, 'Gus', 3, 52000, '2024-02-02');
UPDATE employees SET dept_id = 2 WHERE emp_id = 2;
SELECT dept_id, n FROM dept_headcount ORDER BY dept_id;
SELECT dept_id, COUNT(*) AS real_count FROM employees GROUP BY dept_id ORDER BY dept_id;

dept_id | n
--------+--
1       | 2
2       | 3
3       | 1
(3 rows)

dept_id | real_count
--------+-----------
1       | 2         
2       | 3         
3       | 1         
(3 rows)



## 19. recursive_triggers
- A trigger whose body changes the same table could fire itself again.
- SQLite's `PRAGMA recursive_triggers` is OFF by default: a trigger will not re-fire itself.
- Turn it ON and such a trigger recurses until the data stops changing (or hits the depth limit, default 1000).
- Example: a counter trigger that stops by itself because of its WHEN condition.

In [23]:
%%sql
CREATE TABLE countdown (id INTEGER PRIMARY KEY, n INTEGER NOT NULL);
CREATE TRIGGER trg_countdown AFTER UPDATE ON countdown
WHEN NEW.n > 0
BEGIN
    UPDATE countdown SET n = n - 1 WHERE id = NEW.id;
END;
INSERT INTO countdown VALUES (1, 5), (2, 5);

PRAGMA recursive_triggers;
UPDATE countdown SET n = 4 WHERE id = 1;
SELECT id, n FROM countdown ORDER BY id;

recursive_triggers
------------------
0                 
(1 row)

id | n
---+--
1  | 3
2  | 5
(2 rows)



In [24]:
%%sql
PRAGMA recursive_triggers = ON;
UPDATE countdown SET n = 4 WHERE id = 2;
SELECT id, n FROM countdown ORDER BY id;
PRAGMA recursive_triggers = OFF;

id | n
---+--
1  | 3
2  | 0
(2 rows)



## 20. Gotchas: REPLACE, foreign key actions, and order
- `INSERT OR REPLACE` deletes the old row first. DELETE triggers do NOT fire for that hidden delete unless `recursive_triggers` is ON.
- Deletes caused by `ON DELETE CASCADE` DO fire the child table's DELETE triggers.
- A BEFORE trigger should not assume the row is valid yet; an AFTER trigger sees the final row.
- Several triggers on the same event fire in an unspecified order. Do not depend on it.
- Triggers and constraints both run inside the transaction, so a failure undoes everything.

In [25]:
%%sql
CREATE TABLE pins (code TEXT PRIMARY KEY, label TEXT);
CREATE TABLE pin_deletes (code TEXT);
CREATE TRIGGER trg_pin_del AFTER DELETE ON pins BEGIN INSERT INTO pin_deletes VALUES (OLD.code); END;
INSERT INTO pins VALUES ('a', 'first');

INSERT OR REPLACE INTO pins VALUES ('a', 'second');
SELECT COUNT(*) AS delete_trigger_fired_with_replace FROM pin_deletes;

PRAGMA recursive_triggers = ON;
INSERT OR REPLACE INTO pins VALUES ('a', 'third');
SELECT COUNT(*) AS delete_trigger_fired_with_recursive_on FROM pin_deletes;
PRAGMA recursive_triggers = OFF;

delete_trigger_fired_with_replace
---------------------------------
0                                
(1 row)

delete_trigger_fired_with_recursive_on
--------------------------------------
1                                     
(1 row)



In [26]:
%%sql
CREATE TABLE parents  (parent_id INTEGER PRIMARY KEY);
CREATE TABLE children (child_id INTEGER PRIMARY KEY, parent_id INTEGER REFERENCES parents ON DELETE CASCADE);
CREATE TABLE child_log (child_id INTEGER);
CREATE TRIGGER trg_child_del AFTER DELETE ON children BEGIN INSERT INTO child_log VALUES (OLD.child_id); END;
INSERT INTO parents VALUES (1);
INSERT INTO children VALUES (100, 1), (101, 1);
DELETE FROM parents WHERE parent_id = 1;
SELECT child_id FROM child_log ORDER BY child_id;

child_id
--------
100     
101     
(2 rows)



## 21. Listing and dropping triggers and views
- Both are stored in `sqlite_master` with type 'trigger' or 'view'.
- `DROP TRIGGER [IF EXISTS] name` and `DROP VIEW [IF EXISTS] name`.
- Dropping a table also drops its triggers.
- Change a trigger = drop and recreate (no ALTER TRIGGER).

In [27]:
%%sql
SELECT type, name, tbl_name FROM sqlite_master
WHERE type IN ('view', 'trigger') AND name LIKE 'trg_s%'
ORDER BY type, name;
DROP TRIGGER IF EXISTS trg_salary_rules;
DROP VIEW IF EXISTS big_departments;
SELECT type, name FROM sqlite_master WHERE name IN ('trg_salary_rules', 'big_departments') ORDER BY name;

type    | name                 | tbl_name 
--------+----------------------+----------
trigger | trg_salary_rules     | employees
trigger | trg_salary_update    | employees
trigger | trg_settings_protect | settings 
(3 rows)

type | name
-----+-----
(0 rows)



## 22. Views, triggers, or constraints?
- Rule about one row's own columns -> CHECK / NOT NULL (see 01-Constraints-And-Keys).
- Row must exist in another table -> FOREIGN KEY.
- Rule across rows/tables, history, derived values -> trigger.
- Reusable query shape, hiding columns -> view. Expensive query that must be fast -> summary table (section 13).
- Speed of a view depends on the indexes of the underlying tables (see 07-Performance/01-Indexes-And-Query-Plans).

In [28]:
%%sql
SELECT 'views' AS kind, COUNT(*) AS n FROM sqlite_master WHERE type = 'view'
UNION ALL
SELECT 'triggers', COUNT(*) FROM sqlite_master WHERE type = 'trigger'
ORDER BY kind;

kind     | n 
---------+---
triggers | 14
views    | 4 
(2 rows)

